This is for educational purpose. That is, we format a binary classification task (two classes: 0 and 1) as a regression problem. The model will tend to predict one class (class 0) since that will help minimize MSELoss (all 4 final test examples are classified as negative when trained against all the way up to 9000 examples, tested 10, 1000, 5000, 8000, & 9000) (Interestingly, when trained against 10_000 examples, all 4 final test examples are correctly classified). Plus the decision threshold (0.5 in this implementation) is arbitrary.

In [177]:
import torch
import torch.nn as nn

from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, GPT2Model


In [178]:
# ============================================================
# Configuration
# ============================================================

MODEL_NAME = "openai-community/gpt2"

TRAIN_SIZE = 9000 #10_000
VAL_SIZE = 872

MAX_LENGTH = 128
BATCH_SIZE = 4
NUM_EPOCHS = 1

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01

SEED = 42


if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")


print("Device:", DEVICE)


Device: cuda


In [179]:
# ============================================================
# Reproducibility
# ============================================================

torch.manual_seed(SEED)


In [180]:
# ============================================================
# Load SST-2
# ============================================================

dataset = load_dataset("stanfordnlp/sst2")

print(dataset)

# We use:
#
#   train      -> 10,000 examples
#   validation -> official SST-2 validation set
#
# The official test labels are hidden, so we don't use test here.

train_dataset = (
    dataset["train"]
    .shuffle(seed=SEED)
    .select(range(TRAIN_SIZE))
)

val_dataset = dataset["validation"]


print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

train_labels = torch.tensor(train_dataset["label"])
print("In train, labels: ", train_labels)
print("In train, Class 0: ", (train_labels == 0).sum().item())
print("In train, Class 1: ", (train_labels == 1).sum().item())
print("In train, positive rate: ", train_labels.float().mean().item())

val_labels = torch.tensor(val_dataset["label"])
print("In val, Class 0: ", (val_labels == 0).sum().item())
print("In val, Class 1: ", (val_labels == 1).sum().item())
print("In val, positive rate: ", val_labels.float().mean().item())



DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})
Train size: 9000
Validation size: 872
In train, labels:  tensor([1, 1, 1,  ..., 1, 1, 0])
In train, Class 0:  4062
In train, Class 1:  4938
In train, positive rate:  0.5486666560173035
In val, Class 0:  428
In val, Class 1:  444
In val, positive rate:  0.5091742873191833


In [181]:
# ============================================================
# Tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT-2 does not have a native padding token.
# For batching, use EOS as PAD.
tokenizer.pad_token = tokenizer.eos_token



In [182]:
# ============================================================
# Dataset preprocessing
# ============================================================

def tokenize_example(example):
    encoded = tokenizer(
        example["sentence"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

    encoded["label"] = example["label"]

    return encoded


train_dataset = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
)


val_dataset = val_dataset.map(
    tokenize_example,
    remove_columns=val_dataset.column_names,
)


In [183]:
# ============================================================
# Dynamic padding collator
# ============================================================

def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]

    encoded = tokenizer.pad(
        {
            "input_ids": input_ids,
        },
        padding=True,
        return_tensors="pt",
    )

    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long,
    )

    encoded["labels"] = labels

    return encoded



In [184]:


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)



In [185]:
# ============================================================
# Model
# ============================================================

class GPT2SentimentClassifier(nn.Module):

    def __init__(self, model_name): #, num_labels=2):
        super().__init__()

        # GPT-2 WITHOUT the language-model head.
        self.gpt2 = GPT2Model.from_pretrained(model_name)

        hidden_size = self.gpt2.config.hidden_size

        # Binary regression head:
        #
        # [B, 768] -> [B, 1]
        self.classifier = nn.Linear(
            hidden_size,
            1, #num_labels,
        )


    def forward(self, input_ids, attention_mask, labels=None):

        # ----------------------------------------------------
        # GPT-2
        #
        # last_hidden_state:
        # [B, T, 768]
        # ----------------------------------------------------

        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        hidden_states = outputs.last_hidden_state

        # ----------------------------------------------------
        # Find the position of the last real token
        #
        # attention_mask:
        #
        #   1 1 1 1 1 0 0
        #
        # last real token = position 4

        # ----------------------------------------------------

        sequence_lengths = attention_mask.sum(dim=1) - 1

        batch_indices = torch.arange(
            hidden_states.size(0),
            device=hidden_states.device,
        )

        # [B, 768]
        pooled_hidden_states = hidden_states[
            batch_indices,
            sequence_lengths,
        ]

        # ----------------------------------------------------
        # Regression head
        #
        # [B, 768] -> [B, 1]
        # ----------------------------------------------------

        #logits
        scalars = self.classifier(
            pooled_hidden_states
        )


        scalars = scalars.squeeze(-1)

        ##print(f"In forward => scalars: {scalars}")

        # ----------------------------------------------------
        # Regression loss
        # ----------------------------------------------------

        loss = None

        if labels is not None:
            #print(f"In forward => labels: {labels.float()}")
            #print(f"type of labels: {type(labels)}")
            loss_fn = nn.MSELoss()
            loss = loss_fn(scalars, labels.float()) #(logits, labels)

        #print(f"In forward => loss: {loss}")

        return {
            "loss": loss,
            "scalars": scalars,
            #"logits": logits,
        }



model = GPT2SentimentClassifier(
    MODEL_NAME,
    #num_labels=2,
)



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [186]:
# ============================================================
# Freeze GPT-2
# ============================================================

for parameter in model.gpt2.parameters():
    parameter.requires_grad = False


# Only classifier is trainable.
trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

total_params = sum(
    p.numel()
    for p in model.parameters()
)


trainable_count = sum(
    p.numel()
    for p in trainable_params
)

print()
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_count)





Total parameters: 124440577
Trainable parameters: 769


In [187]:
# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

In [188]:
# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate(model, data_loader):

    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    pred_min = 0.0
    pred_max = 0.0
    pred_mean = 0.0
    pred_sum = 0.0
    pred_gt_threshold_sum = 0.0 # output scalar > 0.5



    for batch in data_loader:

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs["loss"]
        #logits = outputs["logits"]
        scalars = outputs["scalars"]

        #predictions = logits.argmax(dim=-1)
        predictions = (scalars > 0.5).long()


        pred_sum += predictions.float().sum().item()
        if pred_min > predictions.min().item():
            pred_min = predictions.min().item()
        if pred_max < predictions.max().item():
            pred_max = predictions.max().item()

        pred_gt_threshold_sum += (predictions > 0.5).sum().item()

        #print(f"in evaluate => scalars: {scalars}")
        #print(f"in evaluate => predicitons: {predictions}")
        #print(f"in evaluate => labels: {labels}")
        #print(f"in evaluate => loss: {loss.item()}")

        total_loss += loss.item() * labels.size(0)

        total_correct += (
            predictions == labels
        ).sum().item()

        total_examples += labels.size(0)


    print("In val, pred min/max/mean: ", pred_min, pred_max, pred_sum / total_examples)
    print("In val, preds > 0.5: ", pred_gt_threshold_sum / total_examples)

    avg_loss = total_loss / total_examples
    accuracy = total_correct / total_examples

    return avg_loss, accuracy



In [189]:
# ============================================================
# Training
# ============================================================

model.to(DEVICE)

for epoch in range(NUM_EPOCHS):

    model.train()

    running_loss = 0.0

    for step, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        # Forward
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )


        loss = outputs["loss"]

        # Backward
        optimizer.zero_grad()

        loss.backward()

        # Update ONLY the classification head
        optimizer.step()

        running_loss += loss.item()

        if (step + 1) % 100 == 0:

            avg_loss = running_loss / 100

            print(
                f"Epoch {epoch + 1} "
                f"Step {step + 1} "
                f"Loss {avg_loss:.4f}"
            )


            running_loss = 0.0

    val_loss, val_accuracy = evaluate(
        model,
        val_loader,
    )

    print()
    print(
        f"Epoch {epoch + 1} "
        f"Validation Loss: {val_loss:.4f} "
        f"Validation Accuracy: {val_accuracy:.4f}"
    )
    print()



Epoch 1 Step 100 Loss 0.9817
Epoch 1 Step 200 Loss 0.3668
Epoch 1 Step 300 Loss 0.2870
Epoch 1 Step 400 Loss 0.2476
Epoch 1 Step 500 Loss 0.2480
Epoch 1 Step 600 Loss 0.2554
Epoch 1 Step 700 Loss 0.2637
Epoch 1 Step 800 Loss 0.2552
Epoch 1 Step 900 Loss 0.2404
Epoch 1 Step 1000 Loss 0.2133
Epoch 1 Step 1100 Loss 0.2179
Epoch 1 Step 1200 Loss 0.2261
Epoch 1 Step 1300 Loss 0.2492
Epoch 1 Step 1400 Loss 0.2085
Epoch 1 Step 1500 Loss 0.2138
Epoch 1 Step 1600 Loss 0.2040
Epoch 1 Step 1700 Loss 0.1907
Epoch 1 Step 1800 Loss 0.2088
Epoch 1 Step 1900 Loss 0.2114
Epoch 1 Step 2000 Loss 0.2033
Epoch 1 Step 2100 Loss 0.1955
Epoch 1 Step 2200 Loss 0.2158
In val, pred min/max/mean:  0.0 1 0.3532110091743119
In val, preds > 0.5:  0.3532110091743119

Epoch 1 Validation Loss: 0.1546 Validation Accuracy: 0.7936



In [190]:
# ============================================================
# Example predictions
# ============================================================

model.eval()

examples = [
    "This movie was absolutely fantastic.",
    "The movie was boring and disappointing.",
    "I really enjoyed this film.",
    "I wouldn't recommend this movie.",
]

encoded = tokenizer(
    examples,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)


encoded = {
    key: value.to(DEVICE)
    for key, value in encoded.items()
}

with torch.no_grad():

    outputs = model(
        input_ids=encoded["input_ids"],
        attention_mask=encoded["attention_mask"],
    )

    #probabilities = torch.softmax(
    #    outputs["logits"],
    #    dim=-1,
    #)
    probabilities = outputs["scalars"]

    #predictions = probabilities.argmax(dim=-1)
    predictions = (probabilities > 0.5).long()


for text, prediction, probability in zip(
    examples,
    predictions,
    probabilities,
):

    label = (
        "positive"
        if prediction.item() == 1
        else "negative"
    )

    print()
    print("Text:", text)
    print("Prediction:", label)
    print("Probabilities:", probability.cpu().tolist())




Text: This movie was absolutely fantastic.
Prediction: negative
Probabilities: 0.32746759057044983

Text: The movie was boring and disappointing.
Prediction: negative
Probabilities: -0.5677160024642944

Text: I really enjoyed this film.
Prediction: negative
Probabilities: 0.30883467197418213

Text: I wouldn't recommend this movie.
Prediction: negative
Probabilities: -0.2827553153038025
